Click to sample the color and set threshold values

In [ ]:
import cv2
import numpy as np

# -----------------------------
# HARD-CODED CORNERS (your points)
# -----------------------------
pts = np.float32([
    [274, 248],  # TL
    [453, 244],  # TR
    [497, 372],  # BR
    [259, 381]   # BL
])

PADDING = 40

pts_padded = np.float32([
    [pts[0][0] - PADDING, pts[0][1] - PADDING],
    [pts[1][0] + PADDING, pts[1][1] - PADDING],
    [pts[2][0] + PADDING, pts[2][1] + PADDING],
    [pts[3][0] - PADDING, pts[3][1] + PADDING],
])

W = 600
H = 600

dst = np.float32([
    [0, 0],
    [W, 0],
    [W, H],
    [0, H]
])

Hmat, _ = cv2.findHomography(pts_padded, dst)

# -----------------------------
# GLOBALS
# -----------------------------
drawing = False
current_line = None
lines = {}  # top, bottom, left, right
line_order = ["top", "bottom", "left", "right"]
line_index = 0
baseline = None

def nothing(x):
    pass

# -----------------------------
# CLICK & DRAG HANDLER
# -----------------------------
def mouse_event(event, x, y, flags, param):
    global drawing, current_line, lines, line_index

    if line_index >= 4:
        return  # all lines defined

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        current_line = [(x, y)]  # start point

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        if current_line:
            if len(current_line) == 1:
                current_line.append((x, y))
            else:
                current_line[1] = (x, y)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        if current_line and len(current_line) == 2:
            name = line_order[line_index]
            lines[name] = (current_line[0], current_line[1])
            print(f"Defined {name}: {lines[name]}")
            line_index += 1
        current_line = None

# -----------------------------
# BASELINE CALIBRATION
# -----------------------------
def calibrate_baseline(mask):
    global baseline, lines

    baseline = {}

    for name, (p1, p2) in lines.items():
        N = 400
        xs = np.linspace(p1[0], p2[0], N).astype(int)
        ys = np.linspace(p1[1], p2[1], N).astype(int)

        values = []
        for x, y in zip(xs, ys):
            x1 = max(x-1, 0); x2 = min(x+2, W)
            y1 = max(y-1, 0); y2 = min(y+2, H)
            patch = mask[y1:y2, x1:x2]
            values.append(np.any(patch > 0))

        values = np.array(values)
        baseline[name] = np.count_nonzero(values)

    print("\n=== BASELINE CALIBRATED ===")
    print(baseline)

# -----------------------------
# WEBCAM + WINDOWS
# -----------------------------
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

cv2.namedWindow("Warped", cv2.WINDOW_NORMAL)
cv2.namedWindow("Mask", cv2.WINDOW_NORMAL)
cv2.namedWindow("Thresholds", cv2.WINDOW_NORMAL)
cv2.namedWindow("Occlusion Alert", cv2.WINDOW_NORMAL)

cv2.setMouseCallback("Warped", mouse_event)

# -----------------------------
# SLIDERS (your defaults)
# -----------------------------
cv2.createTrackbar("H low",  "Thresholds", 108, 179, nothing)
cv2.createTrackbar("H high", "Thresholds", 179, 179, nothing)
cv2.createTrackbar("S low",  "Thresholds", 0,   255, nothing)
cv2.createTrackbar("S high", "Thresholds", 222, 255, nothing)
cv2.createTrackbar("V low",  "Thresholds", 90,  255, nothing)
cv2.createTrackbar("V high", "Thresholds", 217, 255, nothing)

# -----------------------------
# MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    warped = cv2.warpPerspective(frame, Hmat, (W, H))

    # Read sliders
    hl = cv2.getTrackbarPos("H low", "Thresholds")
    hh = cv2.getTrackbarPos("H high", "Thresholds")
    sl = cv2.getTrackbarPos("S low", "Thresholds")
    sh = cv2.getTrackbarPos("S high", "Thresholds")
    vl = cv2.getTrackbarPos("V low", "Thresholds")
    vh = cv2.getTrackbarPos("V high", "Thresholds")

    lower = np.array([hl, sl, vl])
    upper = np.array([hh, sh, vh])

    hsv = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower, upper)

    warped_overlay = warped.copy()

    # Draw existing lines
    for name, (p1, p2) in lines.items():
        cv2.line(warped_overlay, p1, p2, (0,255,0), 2)
        cv2.putText(warped_overlay, name, (p1[0], p1[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Draw current dragging line
    if current_line and len(current_line) == 2:
        cv2.line(warped_overlay, current_line[0], current_line[1], (0,255,255), 2)

    # Key handling
    key = cv2.waitKey(1)
    if key == ord('c') and len(lines) == 4:
        calibrate_baseline(mask)

    # -----------------------------
    # OCCLUSION DETECTION (robust)
    # -----------------------------
    alert_frame = np.zeros((200, 400, 3), dtype=np.uint8)

    if baseline is not None:
        for name, (p1, p2) in lines.items():
            N = 400
            xs = np.linspace(p1[0], p2[0], N).astype(int)
            ys = np.linspace(p1[1], p2[1], N).astype(int)

            values = []
            for x, y in zip(xs, ys):
                x1 = max(x-1, 0); x2 = min(x+2, W)
                y1 = max(y-1, 0); y2 = min(y+2, H)
                patch = mask[y1:y2, x1:x2]
                values.append(np.any(patch > 0))

            values = np.array(values)  # True = tape present

            # Find breaks (runs of False)
            breaks = []
            run_start = None

            for i, v in enumerate(values):
                if not v and run_start is None:
                    run_start = i
                if v and run_start is not None:
                    if i - run_start > 5:  # break threshold
                        breaks.append((run_start, i))
                    run_start = None

            if run_start is not None and (N - run_start) > 5:
                breaks.append((run_start, N))

            # Highlight breaks
            for (a, b) in breaks:
                bx1, by1 = xs[a], ys[a]
                bx2, by2 = xs[b-1], ys[b-1]
                cv2.line(warped_overlay, (bx1,by1), (bx2,by2), (0,0,255), 3)

            # Trigger alert
            if len(breaks) > 0:
                idx = list(lines.keys()).index(name)
                cv2.putText(alert_frame, f"{name.upper()} BROKEN!",
                            (20, 40 + 40*idx),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    # -----------------------------
    # SHOW WINDOWS
    # -----------------------------
    cv2.imshow("Warped", warped_overlay)
    cv2.imshow("Mask", mask)
    cv2.imshow("Occlusion Alert", alert_frame)

    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()


Defined top: ((112, 147), (503, 152))
Defined bottom: ((79, 504), (537, 498))
Defined left: ((111, 147), (75, 508))
Defined right: ((491, 154), (533, 493))

=== BASELINE CALIBRATED ===
{'top': np.int64(400), 'bottom': np.int64(400), 'left': np.int64(386), 'right': np.int64(400)}
